In [ ]:
# !pip install -q google-genai pydantic
import json
import time
import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

ALLOWED_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other",
]

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"},
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"],
}

MODEL_NAME = "gemini-2.5-flash"  # Update to your Session 22.1 Flash model if different
API_KEY = input("Paste your Gemini API key: ")
client = genai.Client(api_key=API_KEY)
api_call_count = 0

SYSTEM_TEXT = (
    "You are a clinical operations analyst at CareLoop Clinics. "
    "Classify patient no-show follow-up notes accurately into the allowed root_cause categories. "
    "If a note is ambiguous or does not clearly fit a specific category, choose 'other' rather than guessing."
)

def build_craft_prompt(note_text):
    return (
        f"Context: CareLoop Clinics call-centre staff log short notes after calling patients who missed an appointment.\n"
        f"Role: Healthcare operations analyst.\n"
        f"Action: Classify the note below into exactly ONE of these allowed root_cause values: "
        f"{', '.join(ALLOWED_CAUSES)}. If ambiguous, use 'other' rather than guessing.\n"
        f"Tone: Concise, objective, operational.\n"
        f"Format: Return JSON with root_cause, confidence (0 to 1), summary (max 25 words), and recommended_action.\n\n"
        f"Note: {note_text}"
    )

def call_gemini(prompt_text):
    global api_call_count
    api_call_count += 1
    resp = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt_text,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="minimal"),
        ),
    )
    return resp.text

def analyse_note(note_text):
    raw_text = call_gemini(build_craft_prompt(note_text))
    return json.loads(raw_text), raw_text


# ==========================================
# Task 4.1 — One validated call (Note N001)
# ==========================================
notes_df = pd.read_csv("data/followup_notes.csv")
n001_text = notes_df.loc[notes_df["note_id"] == "N001", "note_text"].iloc[0]
n001_data, _ = analyse_note(n001_text)
print("Task 4.1 (N001):", n001_data)
time.sleep(13)


# ==========================================
# Task 4.2 — Validate and process first 10 notes
# ==========================================
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

records = []
for _, row in notes_df.head(10).iterrows():
    nid, aid, text = row["note_id"], row["appointment_id"], row["note_text"]

    # Input validation for blank/NaN notes
    if pd.isna(text) or str(text).strip() == "":
        records.append({
            "note_id": nid, "appointment_id": aid,
            "root_cause": "other", "confidence": 0.0,
            "summary": "No note recorded", "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank", "raw_response": ""
        })
        continue

    raw_text = ""
    data = {}
    status = "invalid"
    try:
        data, raw_text = analyse_note(text)
        insight = FollowupInsight(**data)
        if insight.root_cause in ALLOWED_CAUSES:
            status = "valid"
    except Exception as e:
        print(f"Validation/API error on {nid}: {e}")

    records.append({
        "note_id": nid,
        "appointment_id": aid,
        "root_cause": data.get("root_cause", ""),
        "confidence": data.get("confidence", ""),
        "summary": data.get("summary", ""),
        "recommended_action": data.get("recommended_action", ""),
        "validation_status": status,
        "raw_response": raw_text,
    })
    time.sleep(13)

analysed_df = pd.DataFrame(records)
analysed_df.to_csv("notes_analysed.csv", index=False)
print("\nTask 4.2 Summary:\n", analysed_df[["note_id", "root_cause", "confidence", "validation_status"]])


# ==========================================
# Task 4.3 — Break it on purpose
# ==========================================
print("\n--- Task 4.3(a): Pydantic Validation Tests ---")
bad_cases = [
    {"root_cause": "other", "confidence": 1.7, "summary": "Valid summary", "recommended_action": "Call patient"},
    {"root_cause": "other", "confidence": 0.8, "summary": "Missing action field"},
]
for i, bad in enumerate(bad_cases, 1):
    try:
        FollowupInsight(**bad)
    except ValidationError as e:
        print(f"Bad case {i} error:\n{e}\n")

print("--- Task 4.3(b): Prompt V1 vs V2 on N007 & N009 ---")
compare_rows = []
for nid in ["N007", "N009"]:
    text = notes_df.loc[notes_df["note_id"] == nid, "note_text"].iloc[0]
    # V1: minimal prompt
    raw_v1 = call_gemini(f"Classify this note: {text}")
    d_v1 = json.loads(raw_v1)
    time.sleep(13)
    # V2: reuse CRAFT result from 4.2 to save API calls (total = 12 calls)
    v2_rc = analysed_df.loc[analysed_df["note_id"] == nid, "root_cause"].iloc[0]

    compare_rows.append({"note_id": nid, "prompt": "V1 (minimal)", "root_cause": d_v1.get("root_cause"), "in_allowed": d_v1.get("root_cause") in ALLOWED_CAUSES})
    compare_rows.append({"note_id": nid, "prompt": "V2 (CRAFT)", "root_cause": v2_rc, "in_allowed": v2_rc in ALLOWED_CAUSES})

print(pd.DataFrame(compare_rows))
print(f"\nTask 4.3(c) Total API calls used across Tasks 4.1–4.3: {api_call_count}")


# ==========================================
# Task 4.4(a) — Merge & Cross-tabulate
# ==========================================
clean_df = pd.read_csv("appointments_clean.csv")
merged = analysed_df.merge(clean_df, on="appointment_id", how="left")
print("\nMerged rows:", len(merged))
print("\nCross-tab (root_cause vs reminder_sent):\n", pd.crosstab(merged["root_cause"], merged["reminder_sent"]))